<a href="https://colab.research.google.com/github/Podrimate/THz_sim_application/blob/main/notebooks/THzSim_User_Workflow2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# THzSim2 User Workflow

This notebook is a cleaner end-to-end workflow for THzSim2. The idea is simple:

1. Define the measurement and reference setup.
2. Define the sample and study sweep.
3. Save everything into one setup CSV file.
4. Run the study from that CSV file only.
5. Review the generated summary tables and pairwise heatmaps.

The study workflow now auto-generates pairwise heatmaps for every sweep-axis pair using:

- `normalized_mse`
- `relative_l2`
- `true - fit` for every fitted parameter


## Google Colab

This notebook is designed to work both locally and in Google Colab. In Colab, the runtime does **not** automatically include your local files or installed packages, so the first code cell below includes an install hook.

Recommended sharing options:

- Best one-click workflow: publish the package on PyPI and replace the install spec with `thzsim2`.
- Good workflow today: store the repo on GitHub and install with `!pip install git+https://github.com/<your-user>/<your-repo>.git`.
- If you only upload the notebook file, you can still run it, but you will need to edit the install cell to point to the package source.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
PACKAGE_IMPORT_OK = False
DEFAULT_PIP_SPEC = 'git+https://github.com/Podrimate/THz_sim_application.git'

try:
    import thzsim2  # noqa: F401
    PACKAGE_IMPORT_OK = True
except Exception:
    PACKAGE_IMPORT_OK = False

if not PACKAGE_IMPORT_OK:
    repo_candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in repo_candidates:
        if (candidate / 'thzsim2').exists():
            sys.path.insert(0, str(candidate))
            try:
                import thzsim2  # noqa: F401
                PACKAGE_IMPORT_OK = True
                break
            except Exception:
                pass

if IN_COLAB and not PACKAGE_IMPORT_OK:
    pip_spec = os.environ.get('THZSIM_PIP_SPEC', DEFAULT_PIP_SPEC).strip()
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_spec])
    import thzsim2  # noqa: F401
    PACKAGE_IMPORT_OK = True

if not PACKAGE_IMPORT_OK:
    raise RuntimeError(
        'Could not import thzsim2. Open this notebook inside the repo, or in Colab let the install cell pull from GitHub.'
    )


## Setup / Imports

In [2]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from thzsim2.notebook_api import (
    ConstantNK,
    Drude,
    DrudeLorentz,
    Fit,
    Layer,
    LorentzOscillator,
    NKFile,
    build_study_setup,
    drude_gamma_thz_from_tau_ps,
    drude_plasma_freq_thz_from_sigma_tau,
    load_study_setup_csv,
    run_study_from_setup_csv,
    write_study_setup_csv,
)

workspace_root = Path.cwd()
if not (workspace_root / 'thzsim2').exists() and (workspace_root.parent / 'thzsim2').exists():
    workspace_root = workspace_root.parent

workflow_root = workspace_root / 'notebooks' / 'workflow_outputs'
materials_root = workflow_root / 'materials'
workflow_root.mkdir(parents=True, exist_ok=True)
materials_root.mkdir(parents=True, exist_ok=True)

print('workspace_root =', workspace_root)
print('workflow_root  =', workflow_root)


workspace_root = /content
workflow_root  = /content/notebooks/workflow_outputs


## Helper Functions

In [3]:
def show_setup_sections(setup_csv_path):
    setup = load_study_setup_csv(setup_csv_path)
    for key in ('meta', 'reference', 'sample', 'measurement', 'study', 'notes'):
        if key not in setup:
            continue
        print(f'\n[{key}]')
        print(json.dumps(setup[key], indent=2))


def show_study_heatmaps(study_result, *, contains=None, max_images=None, columns=3):
    heatmaps = []
    for name, path in study_result.artifact_paths.items():
        if not str(name).endswith('_plot'):
            continue
        if contains is not None and contains not in str(name):
            continue
        heatmaps.append((name, Path(path)))
    heatmaps = sorted(heatmaps)
    if max_images is not None:
        heatmaps = heatmaps[:max_images]
    if not heatmaps:
        print('No matching heatmaps found.')
        return
    rows = (len(heatmaps) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(5.5 * columns, 4.5 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (name, path) in zip(axes, heatmaps):
        image = plt.imread(path)
        ax.imshow(image)
        ax.set_title(name.replace('_plot', ''), fontsize=10)
        ax.axis('off')
    for ax in axes[len(heatmaps):]:
        ax.axis('off')
    fig.tight_layout()


def write_example_nk_files(material_dir):
    material_dir = Path(material_dir)
    material_dir.mkdir(parents=True, exist_ok=True)

    freq = np.linspace(0.2, 3.0, 120)

    si_path = material_dir / 'si_example_thz_nk.csv'
    sio2_path = material_dir / 'sio2_backside_example_thz_nk.csv'

    si_n = 3.42 + 0.02 * np.exp(-((freq - 1.8) / 0.9) ** 2)
    si_k = 0.002 + 0.001 * np.exp(-((freq - 2.4) / 0.5) ** 2)

    sio2_n = 1.95 + 0.08 * np.exp(-((freq - 1.2) / 0.35) ** 2)
    sio2_k = 0.18 + 0.12 * np.exp(-((freq - 1.2) / 0.25) ** 2)

    header = 'freq_thz,n,k\n'
    si_rows = '\n'.join(f'{f:.8g},{n:.8g},{k:.8g}' for f, n, k in zip(freq, si_n, si_k))
    sio2_rows = '\n'.join(f'{f:.8g},{n:.8g},{k:.8g}' for f, n, k in zip(freq, sio2_n, sio2_k))

    si_path.write_text(header + si_rows + '\n', encoding='utf-8')
    sio2_path.write_text(header + sio2_rows + '\n', encoding='utf-8')
    return si_path, sio2_path


## 1. Measurement And Reference Setup

Start here. Choose the measurement geometry and whether the reference comes from a measured CSV or from a generated pulse.


In [4]:
use_measured_reference = False
reference_csv_path = workflow_root / 'reference_trace.csv'
measurement_mode = 'transmission'  # 'transmission' or 'reflection'
incident_angle_deg = 25.0
polarization = 'p'  # 's' or 'p'

if use_measured_reference:
    reference_setup = {
        'kind': 'measured_csv',
        'path': reference_csv_path,
        'time_column': None,
        'signal_column': None,
        'prepare': {
            'output_root': workflow_root / 'runs',
            'run_label': 'user-workflow-measured-reference',
            'noise': None,
        },
    }
else:
    reference_setup = {
        'kind': 'generated_pulse',
        'generate': {
            'model': 'sech_carrier',
            'sample_count': 2048,
            'dt_ps': 0.02,
            'time_center_ps': 12.0,
            'pulse_center_ps': 6.5,
            'tau_ps': 0.28,
            'f0_thz': 0.9,
            'amp': 1.0,
            'phi_rad': 0.0,
            'pad_factor': 1,
        },
        'prepare': {
            'output_root': workflow_root / 'runs',
            'run_label': 'user-workflow-generated-reference',
            'noise': None,
        },
    }

measurement_setup = {
    'mode': measurement_mode,
    'angle_deg': incident_angle_deg,
    'polarization': polarization,
    'reference_standard': {'kind': 'identity'},
}

if measurement_mode == 'reflection':
    measurement_setup['reference_standard'] = {
        'kind': 'stack',
        'stack': {
            'layers': [
                {
                    'name': 'substrate_only',
                    'thickness_um': 500.0,
                    'material': {'kind': 'ConstantNK', 'n': 1.55, 'k': 0.0},
                }
            ],
            'n_in': 1.0,
            'n_out': 1.0,
        },
    }

print(json.dumps(measurement_setup, indent=2))


{
  "mode": "transmission",
  "angle_deg": 25.0,
  "polarization": "p",
  "reference_standard": {
    "kind": "identity"
  }
}


## 2. Arbitrary Sample And Study Sweep

Define the sample with any combination of supported material models. Then define the `study` truth sweep and the fit settings.


In [5]:
sample_layers = [
    Layer(
        name='coating',
        thickness_um=Fit(120.0, abs_min=70.0, abs_max=170.0, label='coating_thickness_um'),
        material=ConstantNK(
            n=Fit(2.15, abs_min=1.8, abs_max=2.5, label='coating_n'),
            k=Fit(0.03, abs_min=0.0, abs_max=0.08, label='coating_k'),
        ),
    ),
    Layer(
        name='substrate',
        thickness_um=500.0,
        material=ConstantNK(n=1.55, k=0.0),
    ),
]

study_config = {
    'truth': {
        'layers[0].thickness_um': [90.0, 120.0, 150.0],
        'layers[0].material.n': [2.0, 2.15, 2.3],
        'layers[0].material.k': [0.01, 0.03, 0.05],
    },
    'noise_dynamic_range_db': [55.0, 70.0],
    'replicates': 1,
    'seed': 123,
    'metric': 'mse',
    'optimizer': {
        'method': 'L-BFGS-B',
        'options': {'maxiter': 20},
        'global_options': {'maxiter': 2, 'popsize': 5, 'seed': 123},
        'fd_rel_step': 1e-4,
    },
    'out_dir': workflow_root / 'study_outputs' / 'main_study',
}

sample_options = {
    'n_in': 1.0,
    'n_out': 1.0,
    'overlay_imported': True,
    'sample_out_dir': workflow_root / 'sample_builds' / 'main_sample',
}


## 3. Save Everything Into One CSV Setup File

This is the portable study definition. The file contains the reference setup, sample definition, measurement settings, and study sweep.


In [6]:
setup = build_study_setup(
    reference=reference_setup,
    layers=sample_layers,
    measurement=measurement_setup,
    study=study_config,
    n_in=sample_options['n_in'],
    n_out=sample_options['n_out'],
    overlay_imported=sample_options['overlay_imported'],
    sample_out_dir=sample_options['sample_out_dir'],
    notes='Main user workflow setup',
)

setup_csv_path = workflow_root / 'study_setup_main.csv'
write_study_setup_csv(setup_csv_path, setup)
print('setup_csv_path =', setup_csv_path)
show_setup_sections(setup_csv_path)


setup_csv_path = /content/notebooks/workflow_outputs/study_setup_main.csv

[meta]
{
  "schema_version": 1,
  "created_at": "2026-04-19T13:31:59.141713+00:00"
}

[reference]
{
  "kind": "generated_pulse",
  "generate": {
    "model": "sech_carrier",
    "sample_count": 2048,
    "dt_ps": 0.02,
    "time_center_ps": 12.0,
    "pulse_center_ps": 6.5,
    "tau_ps": 0.28,
    "f0_thz": 0.9,
    "amp": 1.0,
    "phi_rad": 0.0,
    "pad_factor": 1
  },
  "prepare": {
    "output_root": "/content/notebooks/workflow_outputs/runs",
    "run_label": "user-workflow-generated-reference",
    "noise": null
  }
}

[sample]
{
  "layers": [
    {
      "name": "coating",
      "thickness_um": {
        "kind": "Fit",
        "initial": 120.0,
        "rel_min": null,
        "rel_max": null,
        "abs_min": 70.0,
        "abs_max": 170.0,
        "label": "coating_thickness_um"
      },
      "material": {
        "kind": "ConstantNK",
        "n": {
          "kind": "Fit",
          "initial": 2.1

## 4. Run The Study From The CSV File Only

From this point onward, the study can be regenerated from the CSV setup file by itself.


In [7]:
study_result = run_study_from_setup_csv(setup_csv_path)
print('study out dir:', study_result.out_dir)
print('summary csv  :', study_result.summary_csv_path)
print('case count    :', len(study_result.summary_rows))


study out dir: /content/notebooks/workflow_outputs/study_outputs/main_study
summary csv  : /content/notebooks/workflow_outputs/study_outputs/main_study/study_summary.csv
case count    : 54


## 5. Review The Generated Outputs

In [8]:
summary_rows = study_result.summary_rows
print('first row keys:', list(summary_rows[0].keys())[:20])
for row in summary_rows[:3]:
    print({
        'case_id': row['case_id'],
        'normalized_mse': row['normalized_mse'],
        'relative_l2': row['relative_l2'],
        'true__coating_thickness_um': row.get('true__coating_thickness_um'),
        'fit__coating_thickness_um': row.get('fit__coating_thickness_um'),
    })


first row keys: ['case_id', 'replicate_id', 'seed', 'success', 'objective_value', 'mse', 'normalized_mse', 'windowed_mse', 'relative_l2', 'peak_normalized_rmse', 'snr_db', 'max_abs_parameter_correlation', 'mean_abs_parameter_correlation', 'measurement_mode', 'measurement_angle_deg', 'measurement_polarization', 'layers[0].thickness_um', 'layers[0].material.n', 'layers[0].material.k', 'noise_dynamic_range_db']
{'case_id': 0, 'normalized_mse': 4.1708768971168835e-06, 'relative_l2': 0.023749138400438137, 'true__coating_thickness_um': 90.0, 'fit__coating_thickness_um': 85.6921230188996}
{'case_id': 1, 'normalized_mse': 1.0579216498102083e-06, 'relative_l2': 0.011955618035870369, 'true__coating_thickness_um': 90.0, 'fit__coating_thickness_um': 85.69213043228251}
{'case_id': 2, 'normalized_mse': 4.07289113019756e-06, 'relative_l2': 0.02340549062915288, 'true__coating_thickness_um': 90.0, 'fit__coating_thickness_um': 85.69211395844674}


In [9]:
show_study_heatmaps(study_result, contains='normalized-mse__', max_images=6)


In [10]:
show_study_heatmaps(study_result, contains='relative-l2__', max_images=6)


In [11]:
show_study_heatmaps(study_result, contains='signed-err', max_images=9)


## Optional Scenario Templates

These helpers provide more advanced starting points. They are not run automatically.


In [12]:
def make_single_layer_drude_template(output_root):
    gamma_grid = [drude_gamma_thz_from_tau_ps(tau) for tau in (0.2, 0.5, 1.0)]
    plasma_grid = [drude_plasma_freq_thz_from_sigma_tau(sig, 0.5) for sig in (2e3, 5e3, 1e4)]
    layers = [
        Layer(
            name='drude_layer',
            thickness_um=Fit(60.0, abs_min=20.0, abs_max=120.0, label='drude_thickness_um'),
            material=Drude(
                eps_inf=12.0,
                plasma_freq_thz=Fit(0.4, abs_min=0.1, abs_max=1.2, label='drude_plasma_freq_thz'),
                gamma_thz=Fit(0.4, abs_min=0.05, abs_max=2.0, label='drude_gamma_thz'),
            ),
        )
    ]
    study = {
        'truth': {
            'layers[0].thickness_um': [30.0, 60.0, 90.0],
            'layers[0].material.plasma_freq_thz': plasma_grid,
            'layers[0].material.gamma_thz': gamma_grid,
        },
        'noise_dynamic_range_db': [55.0, 70.0],
        'replicates': 1,
        'seed': 123,
        'optimizer': {
            'method': 'L-BFGS-B',
            'options': {'maxiter': 20},
            'global_options': {'maxiter': 2, 'popsize': 5, 'seed': 123},
            'fd_rel_step': 1e-4,
        },
        'out_dir': Path(output_root) / 'drude_template_study',
    }
    measurement = {'mode': 'transmission', 'angle_deg': 0.0, 'polarization': 's', 'reference_standard': {'kind': 'identity'}}
    return layers, measurement, study


def make_two_layer_lorentz_si_template(output_root, material_dir):
    si_path, _ = write_example_nk_files(material_dir)
    layers = [
        Layer(
            name='lorentz_film',
            thickness_um=Fit(30.0, abs_min=10.0, abs_max=60.0, label='film_thickness_um'),
            material=DrudeLorentz(
                eps_inf=Fit(2.2, abs_min=1.5, abs_max=3.5, label='film_eps_inf'),
                plasma_freq_thz=0.0,
                gamma_thz=0.0,
                oscillators=(
                    LorentzOscillator(
                        delta_eps=Fit(2.8, abs_min=1.0, abs_max=5.0, label='osc1_delta_eps'),
                        resonance_thz=Fit(0.8, abs_min=0.4, abs_max=1.3, label='osc1_resonance_thz'),
                        gamma_thz=Fit(0.15, abs_min=0.05, abs_max=0.5, label='osc1_gamma_thz'),
                    ),
                    LorentzOscillator(
                        delta_eps=Fit(1.6, abs_min=0.5, abs_max=4.0, label='osc2_delta_eps'),
                        resonance_thz=Fit(1.8, abs_min=1.1, abs_max=2.5, label='osc2_resonance_thz'),
                        gamma_thz=Fit(0.22, abs_min=0.08, abs_max=0.7, label='osc2_gamma_thz'),
                    ),
                ),
            ),
        ),
        Layer(
            name='si_substrate',
            thickness_um=Fit(450.0, abs_min=300.0, abs_max=700.0, label='si_thickness_um'),
            material=NKFile(si_path),
        ),
    ]
    study = {
        'truth': {
            'layers[0].thickness_um': [20.0, 30.0],
            'layers[0].material.eps_inf': [2.0, 2.4],
            'layers[0].material.oscillators[0].delta_eps': [2.4, 3.0],
            'layers[0].material.oscillators[0].resonance_thz': [0.75, 0.9],
            'layers[0].material.oscillators[1].delta_eps': [1.2, 1.8],
            'layers[0].material.oscillators[1].resonance_thz': [1.6, 1.95],
            'layers[1].thickness_um': [420.0, 520.0],
        },
        'noise_dynamic_range_db': [60.0, 75.0],
        'replicates': 1,
        'seed': 321,
        'optimizer': {
            'method': 'L-BFGS-B',
            'options': {'maxiter': 25},
            'global_options': {'maxiter': 2, 'popsize': 6, 'seed': 321},
            'fd_rel_step': 1e-4,
        },
        'out_dir': Path(output_root) / 'lorentz_si_template_study',
    }
    measurement = {'mode': 'transmission', 'angle_deg': 15.0, 'polarization': 's', 'reference_standard': {'kind': 'identity'}}
    return layers, measurement, study


def make_three_layer_backside_template(output_root, material_dir):
    si_path, sio2_path = write_example_nk_files(material_dir)
    layers = [
        Layer(
            name='lorentz_film',
            thickness_um=Fit(28.0, abs_min=10.0, abs_max=55.0, label='film_thickness_um'),
            material=DrudeLorentz(
                eps_inf=Fit(2.4, abs_min=1.5, abs_max=3.5, label='film_eps_inf'),
                plasma_freq_thz=0.0,
                gamma_thz=0.0,
                oscillators=(
                    LorentzOscillator(
                        delta_eps=Fit(2.6, abs_min=1.0, abs_max=5.0, label='osc1_delta_eps'),
                        resonance_thz=Fit(0.85, abs_min=0.4, abs_max=1.3, label='osc1_resonance_thz'),
                        gamma_thz=Fit(0.16, abs_min=0.05, abs_max=0.5, label='osc1_gamma_thz'),
                    ),
                    LorentzOscillator(
                        delta_eps=Fit(1.4, abs_min=0.5, abs_max=4.0, label='osc2_delta_eps'),
                        resonance_thz=Fit(1.9, abs_min=1.1, abs_max=2.6, label='osc2_resonance_thz'),
                        gamma_thz=Fit(0.24, abs_min=0.08, abs_max=0.7, label='osc2_gamma_thz'),
                    ),
                ),
            ),
        ),
        Layer(
            name='si_substrate',
            thickness_um=Fit(500.0, abs_min=350.0, abs_max=750.0, label='si_thickness_um'),
            material=NKFile(si_path),
        ),
        Layer(
            name='sio2_backside',
            thickness_um=50000.0,
            material=NKFile(sio2_path),
        ),
    ]
    study = {
        'truth': {
            'layers[0].thickness_um': [22.0, 32.0],
            'layers[0].material.eps_inf': [2.0, 2.6],
            'layers[0].material.oscillators[0].delta_eps': [2.2, 2.8],
            'layers[0].material.oscillators[1].resonance_thz': [1.7, 2.05],
            'layers[1].thickness_um': [450.0, 550.0],
        },
        'noise_dynamic_range_db': [60.0, 75.0],
        'replicates': 1,
        'seed': 456,
        'optimizer': {
            'method': 'L-BFGS-B',
            'options': {'maxiter': 25},
            'global_options': {'maxiter': 2, 'popsize': 6, 'seed': 456},
            'fd_rel_step': 1e-4,
        },
        'out_dir': Path(output_root) / 'lorentz_si_sio2_template_study',
    }
    measurement = {'mode': 'transmission', 'angle_deg': 20.0, 'polarization': 'p', 'reference_standard': {'kind': 'identity'}}
    return layers, measurement, study


## Sharing This Notebook In Colab

Practical options:

- If your repo is on GitHub: open the notebook in Colab from GitHub, set the install cell to `git+https://github.com/<your-user>/<your-repo>.git`, then share the Colab notebook with the normal **Share** button.
- If you publish the package on PyPI: change the install cell to `pip install thzsim2` and the notebook becomes closer to one-click for everyone.
- If you want zero manual install edits for other users, the package URL has to be stable somewhere public, because Colab shares notebook content but not the runtime environment or custom installed files.
